# TP2 - Informe tecnico

Sistema de Deteccion y Clasificacion de Razas de Perros — IA 5.2 Computer Vision.

## Equipo
- Alumno 1 : Calabozo Nicolás
- Alumno 2 : Lapolla Martín

## 1. Explicacion completa del pipeline

TO-DO: describir el flujo Embeddings -> Busqueda por similitud -> Clasificacion -> Deteccion -> Pipeline completo,
y como se integran los componentes (base vectorial, modelos, YOLO, aplicacion Gradio).

El pipeline completo consta de tres etapas:

- Etapa 1 **Busqueda por Similitud**: Se extrae un embedding de la imagen consultada con un modelo pre-entrenado, se buscan en una base vectorial (PostgreSQL + pgvector) los vecinos mas cercanos y se predice la raza

- Etapa 2 **Clasificacion**: Se entrena un clasificador especifico para las 70 razas. Modelo A: ResNet18 fine-tuned con transfer learning. Modelo B: CustomCNN con bloques residuales entrenado desde cero.

- Etapa 3 **Deteccion + Clasificacion**: YOLOv8m detecta perros en la imagen. Cada recorte se clasifica con uno de los modelos entrenados en Etapa 2, el que haya sido seleccionado. El resultado combina la ubicacion del bounding box con la raza predicha y su confianza.


## 2. Dataset

TO-DO: distribucion de clases, cantidad de imagenes por raza, definicion de splits
(train/valid/test) y conjunto independiente de evaluacion.

Se utiliza el dataset 70 Dog Breeds Image Dataset (Kaggle), que ya cuenta con un split train/valid/test hecho previamente sobre las casi 9300 imágenes. Las categorias presentes están desbalanceadas, teiendo algunas razas cerca de 200 imágenes mientras que las que menso presentan cerca de 60.

### SE PODRÍA PONER DEBAJO CÓDIGO PARA MOSTRAR EL DESBALANCE

## 3. Preprocesamiento

TO-DO: tecnicas aplicadas (resize, normalizacion, data augmentation, filtrado) y su justificacion.

## 4. Justificacion de los modelos elegidos

TO-DO: modelo de embeddings baseline (Etapa 1), ResNet18 fine-tuned y CNN custom (Etapa 2),
YOLO (Etapa 3). Analizar trade-offs: precision, velocidad de inferencia, consumo de memoria
y complejidad computacional.

Etapa 1 - **Embeddings baseline**:
- Para la etapa 1, el modelo elegido es EfficientNet-B0, pre-entrenado en ImageNet. Se utiliza este modelo para la generación de embeddings debido a su precisión similar o superior a redes mucho más grandes (como ResNet-50) pero con una fracción del costo computacional con 5,3 millones de parámetros. 


Etapa 2 - **Modelo A (ResNet18 fine-tuned)**:
- Elejimos utilizar ResNet18 y no otro modelo más grande de la familia ResNet porque al estar haciendo transfer learning la diferencia de profundidad con los otros modelos no va a impactar tanto en la extracción de características. Las capas convolucionales del modelo ya están preentrenadas en ImageNet, en todos los casos, y capturan patrones generales, bordes, texturas, formas, que son suficientes para llevar a cabo la distinción entre razas. Además al contar con muchos menos parámetros que los otros modelos reducimos las posibilidades de overfitting al trabajar con dataset pequeño en relación con el que fue preentrenado ResNet.   

Etapa 2 - **Modelo B (CNN propia)**:
- Creamos una CNN propia con 4 bloques residuales (canales 64->128->256->512), convolución inicial de 7×7 con stride 2, seguida de MaxPooling, Global Average Pooling, Dropout(0.4) y capa lineal final para 70 clases. El diseño incorpora shortcuts en cada bloque para preservar la información y facilitar el flujo de los gradientes, siguiendo una arquitectura muy similar a ResNet. El modelo fue entrenado desde cero sin pesos preentrenados. La elección de una arquitectura moderada en profundidad responde a la necesidad de equilibrar capacidad de representación y limitaciones de cómputo disponibles, evitando un diseño más complejo que hubiera incrementado el costo de entrenamiento. Aun así, esta CNN sirve como baseline de comparación frente a modelos preentrenados, permitiendo demostrar el impacto positivo del transfer learning en tareas de clasificación de imágenes.

Etapa 3 - **YOLOv8m**:
-  Se eligio el modelo medium de la familia YOLOv8, pre-entrenado en COCO (que ya incluye la clase 'dog'). La idea detrás de la elección de este tamaño de modelo viene dada por la velocidad del modelo para hacer inferencia y su capacidad de detectar todos los perros presentes en las distintas imágenes con buenos scores de confianza.

## 5. Proceso de entrenamiento e hiperparametros

TO-DO: proceso de fine-tuning, hiperparametros utilizados (learning rate, batch size, epochs,
optimizador, scheduler), curvas de entrenamiento.

## 6. Resultados obtenidos

TO-DO:
- Etapa 1: NDCG@10 y justificacion del resultado.
- Etapa 2: accuracy, precision, recall, specificity, F1, matriz de confusion.

## 7. Comparacion entre enfoques

TO-DO: busqueda por similitud vs clasificacion supervisada; ResNet18 fine-tuned vs CNN custom.

## 8. Problemas encontrados y soluciones implementadas

TO-DO.

## 9. Modificaciones fuera de las funciones indicadas

TO-DO: justificar debidamente cualquier cambio realizado fuera de las funciones
indicadas en cada etapa (si no hubo, indicarlo).